In [1]:
#| hide

import inspect
from IPython.display import display, Markdown

from peshbeen.models import (
    naive,
    ets,
    arima,
    auto_arima,
    ml_forecaster,
    ml_direct_forecaster,
    glm,
    ms_arr,
    var,
    ms_var,
    panel_forecaster,
    ml_multi_forecaster,
    pesh,
)
#| hide
from fastcore.docments import docments, DocmentTbl
from nbdev.showdoc import *

def render_api(obj):
    """
    Generates a clean, Sphinx-style API reference with multi-line 
    arguments and deduplicated module paths.
    """
    # 1. Get module and class/function names
    mod = getattr(obj, '__module__', '')
    qualname = getattr(obj, '__qualname__', obj.__name__)
    
    # Clean up __init__ to just show the class name
    is_init = obj.__name__ == "__init__"
    if is_init:
        qualname = qualname.replace(".__init__", "")
        base_name = qualname.split(".")[-1]
        func_name = f"{base_name}"
    else:
        base_name = obj.__name__
        func_name = f"{base_name}"
        
    # 2. Deduplicate the path (fixes peshbeen.models.ml_forecaster.ml_forecaster)
    path_parts = mod.split('.') + qualname.split('.')
    clean_parts = []
    for part in path_parts:
        if not clean_parts or clean_parts[-1] != part:
            clean_parts.append(part)
            
    clean_path = ".".join(clean_parts)
    
    # 3. Extract the exact Python signature and format it line-by-line
    try:
        sig = inspect.signature(obj)
        params = []
        for name, param in sig.parameters.items():
            if name == "self":
                continue  # Hide 'self' from the documentation
            params.append(f"    {param}")
            
        if params:
            param_str = ",\n".join(params)
            sig_str = f"{func_name}(\n{param_str}\n)"
        else:
            sig_str = f"{func_name}()"
    except ValueError:
        sig_str = func_name
        
    # 4. Extract just the main description from your docstring
    doc = inspect.getdoc(obj) or ""
    description = doc.split("Parameters")[0].strip()
    
    # 5. Build the exact Markdown structure and render
    md_content = f"### {clean_path}\n\n```python\n{sig_str}\n```\n\n{description}"
    
    display(Markdown(md_content))
    display(DocmentTbl(obj))


# Forecasters

## Univariate forecasters

### Naive forecaster

In [2]:
#| echo: false
render_api(naive.__init__)
render_api(naive.fit)
render_api(naive.forecast)
render_api(naive.cross_validate)

### peshbeen.models.naive

```python
naive(
    target_col: 'str',
    season_period: 'Optional[int]' = None,
    box_cox: 'Union[bool, float]' = False,
    box_cox_biasadj: 'bool' = False
)
```

Naïve forecaster.

Two modes controlled by ``season_period``:

* **Non-seasonal** (``season_period=None``): every forecast step repeats
the last observed value in the training series.
* **Seasonal** (``season_period=m``): forecast values are taken from the
last complete season and cycled forward — i.e. step ``h`` is predicted
by ``y[T - m + ((h-1) % m)]``, where ``T`` is the last training index.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| target_col | str |  | Name of the target variable column. |
| season_period | Optional[int] | None | Seasonal period ``m``.  ``None`` selects the non-seasonal naïve method.  When provided and the training series is shorter than ``m``, ``forecast`` returns an array of ``NaN``. |
| box_cox | Union[bool, float] | False | Whether to apply Box-Cox transformation to the target variable. If a float value is provided, it will be used as the lambda parameter for the Box-Cox transformation. If True, the lambda parameter will be estimated from the data. |
| box_cox_biasadj | bool | False | Bias adjustment when inverting the manual Box-Cox on forecasts. |
| **Returns** | **None** |  |  |

### peshbeen.models.naive.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Store the values needed for naïve forecasting.

No statistical model is estimated.  ``fit`` simply applies
``data_prep`` and records the training series so that ``forecast``
can replicate the correct naïve pattern.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target column. |
| **Returns** | **None** |  |

### peshbeen.models.naive.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Generate naïve forecasts.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. |
| exog | Optional[pd.DataFrame] | None | Accepted for API consistency with other models but silently ignored — naïve forecasts do not use exogenous variables. |
| **Returns** | **np.ndarray** |  | **Forecast values of length `H`.** |

### peshbeen.models.naive.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run time-series cross-validation.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | Full dataset. |
| cv_split | int |  | Number of CV folds. |
| test_size | int |  | Test window size per fold. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size to advance the test window each fold. |
| h_split_point | Optional[int] | None | Split the test window into two sub-horizons for separate short- and long-term evaluation. |
| **Returns** | **Tuple[pd.DataFrame, pd.DataFrame]** |  | **Summary DataFrame with mean metric scores across folds, and (optionally) a fold-level DataFrame with true vs. predicted values for each fold.** |

### ETS (Error, Trend, Seasonality) forecaster

In [3]:
#| echo: false
render_api(ets.__init__)
render_api(ets.fit)
render_api(ets.forecast)
render_api(ets.cross_validate)

### peshbeen.models.ets

```python
ets(
    target_col: 'str',
    trend: 'Optional[str]' = None,
    damped_trend: 'bool' = False,
    seasonal: 'Optional[str]' = None,
    seasonal_periods: 'Optional[int]' = None,
    initialization_method: 'Optional[str]' = 'estimated',
    initial_level: 'Optional[float]' = None,
    initial_trend: 'Optional[float]' = None,
    initial_seasonal: 'Optional[list]' = None,
    bounds: 'Optional[dict]' = None,
    dates=None,
    freq: 'Optional[str]' = None,
    missing: 'str' = 'none',
    optimized: 'bool' = True,
    smoothing_level: 'Optional[float]' = None,
    smoothing_trend: 'Optional[float]' = None,
    smoothing_seasonal: 'Optional[float]' = None,
    damping_trend: 'Optional[float]' = None,
    remove_bias: 'bool' = False,
    start_params=None,
    method: 'Optional[str]' = None,
    minimize_kwargs: 'Optional[dict]' = None,
    use_brute: 'bool' = True,
    box_cox: 'Union[bool, float]' = False,
    box_cox_biasadj: 'bool' = False,
    fit_kwargs: 'Optional[dict]' = None
)
```

Holt-Winters Exponential Smoothing forecaster.

A thin wrapper around ``statsmodels.tsa.holtwinters.ExponentialSmoothing``.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| target_col | str |  | Name of the target variable column. |
| trend | Optional[str] | None | Trend component type. |
| damped_trend | bool | False | Whether to damp the trend.  Only meaningful when ``trend`` is not `None`. |
| seasonal | Optional[str] | None | Seasonal component type. |
| seasonal_periods | Optional[int] | None | Number of periods in a complete seasonal cycle — e.g. 12 for monthly data with an annual cycle.  Required when ``seasonal`` is not ``None``. |
| initialization_method | Optional[str] | estimated | How to initialise the recursions.  When `"known"` is chosen, `initial_level` (and `initial_trend` / `initial_seasonal` where applicable) must also be provided. |
| initial_level | Optional[float] | None | Initial level value.  Required when `initialization_method=`"known"`. |
| initial_trend | Optional[float] | None | Initial trend value.  Required when `initialization_method=`"known"` and the model has a trend component. |
| initial_seasonal | Optional[list] | None | Initial seasonal factors (length `seasonal_periods` or ``seasonal_periods - 1``).  Required when ``initialization_method="known"`` and the model is seasonal. |
| bounds | Optional[dict] | None | Parameter bounds passed to ``ExponentialSmoothing``, e.g. `{"smoothing_level": (0, 1)}`. |
| dates | NoneType | None | Datetime index for the series.  Inferred automatically when `endog` is a Pandas object with a `DatetimeIndex`. |
| freq | Optional[str] | None | Frequency of the time series (e.g. ``"M"``, ``"D"``).  Optional when `dates` is provided. |
| missing | str | none | How to handle ``NaN`` values in the input series. |
| optimized | bool | True | Estimate smoothing parameters by maximising the log-likelihood. |
| smoothing_level | Optional[float] | None | Fixed alpha value.  When set, this value is used directly and not optimised. |
| smoothing_trend | Optional[float] | None | Fixed beta value.  Only used when the model has a trend component. |
| smoothing_seasonal | Optional[float] | None | Fixed gamma value.  Only used when the model is seasonal. |
| damping_trend | Optional[float] | None | Fixed phi (damping) value.  Only used when ``damped_trend=True``. |
| remove_bias | bool | False | Remove bias from forecast values by enforcing that the mean residual is zero. |
| start_params | NoneType | None | Starting parameter values for the optimiser. |
| method | Optional[str] | None | Optimisation method — one of `"L-BFGS-B"` (default), `"TNC"`, `"SLSQP"`, `"Powell"`, `"trust-constr"`, `"basinhopping"` (alias `"bh"`), or `"least_squares"` (alias `"ls"`). |
| minimize_kwargs | Optional[dict] | None | Extra keyword arguments forwarded to the chosen SciPy minimiser. |
| use_brute | bool | True | Search for good starting values with a brute-force grid search before running the main optimiser. |
| box_cox | Union[bool, float] | False | Whether to apply Box-Cox transformation to the target variable. If a float value is provided, it will be used as the lambda parameter for the Box-Cox transformation. If True, the lambda parameter will be estimated from the data. |
| box_cox_biasadj | bool | False | Bias adjustment when inverting the manual Box-Cox on forecasts. |
| fit_kwargs | Optional[dict] | None | Any additional keyword arguments forwarded verbatim to `ExponentialSmoothing.fit`. |
| **Returns** | **None** |  | **Fitted model object with a ``forecast`` method for making predictions and properties for information criteria scores (AIC, BIC, etc.)** |

### peshbeen.models.ets.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit ``ExponentialSmoothing`` to the training data.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target column |
| **Returns** | **None** |  |

### peshbeen.models.ets.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Multi-step forecast.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. |
| exog | Optional[pd.DataFrame] | None | Accepted for API consistency with other models but silently ignored — ETS forecasts do not use exogenous variables. |
| **Returns** | **np.ndarray** |  | **Forecast values of length `H`.** |

### peshbeen.models.ets.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run time-series cross-validation.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | Full dataset |
| cv_split | int |  | Number of CV folds. |
| test_size | int |  | Test window size per fold. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size to advance the test window each fold. |
| h_split_point | Optional[int] | None | Split the test window into two sub-horizons for separate short- and long-term evaluation. |
| **Returns** | **Tuple[pd.DataFrame, pd.DataFrame]** |  | **Summary DataFrame with mean metric scores across folds, and (optionally) a fold-level DataFrame with true vs. predicted values for each fold.** |

### ARIMA forecaster

In [4]:
#| echo: false
render_api(arima.__init__)
render_api(arima.fit)
render_api(arima.forecast)
render_api(arima.cross_validate)

### peshbeen.models.arima

```python
arima(
    target_col: 'str',
    order: 'Optional[Tuple[int, int, int]]' = (0, 0, 0),
    seasonal_order: 'Optional[Tuple[int, int, int]]' = (0, 0, 0),
    seasonal_length: 'Optional[int]' = None,
    lag_transform: 'Optional[list]' = None,
    trend: 'Optional[str]' = None,
    pol_degree: 'int' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[List[int]]' = None,
    box_cox: 'Union[bool, float, int]' = False,
    box_cox_biasadj: 'bool' = False,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    target_encode: 'bool' = False,
    include_constant: 'bool' = False,
    maxiter: 'int' = 200,
    tol: 'float' = 1e-07
)
```

Initialize the arima model with the specified parameters and configurations.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| target_col | str |  | Name of the target variable column in the input DataFrame. |
| order | Optional[Tuple[int, int, int]] | (0, 0, 0) | The (p, d, q) order of the ARIMA model. Default is (0, 0, 0). |
| seasonal_order | Optional[Tuple[int, int, int]] | (0, 0, 0) | The (P, D, Q) order of the seasonal ARIMA model, where P is the seasonal AR order, D is the seasonal differencing order, and Q is the seasonal MA order. Default is (0, 0, 0). |
| seasonal_length | Optional[int] | None | The seasonal period length (s) for the seasonal ARIMA model. Default is None. |
| lag_transform | Optional[list] | None | List of lag-transform function objects to apply to the target variable (e.g. [expanding_mean(shift=1), rolling_std(window_size=3, shift=1)]). Each function should take a pandas Series as input and return a Series of the same length. Default is None (no lag transforms). |
| trend | Optional[str] | None | Trend strategy to use. Options are 'linear' for linear trend removal and 'ets' for ETS-based trend removal. Default is None (no trend handling). |
| pol_degree | int | 1 | Degree of polynomial trend to fit when using 'linear' trend strategy. Default is 1 (linear trend). |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary of parameters for the ExponentialSmoothing model when using 'ets' trend strategy. The keys should be the parameter names and the values should be the parameter values. Default is None (use default ETS parameters). |
| change_points | Optional[List[int]] | None | List of indices in the time series where change points occur for piecewise linear trend fitting. Only used when trend strategy is 'linear'. Default is None (no change points, fit a single linear trend). |
| box_cox | Union[bool, float, int] | False | Whether to apply Box-Cox transformation to the target variable. If a float or int value is provided, it will be used as the lambda parameter for the Box-Cox transformation. If True, the lambda parameter will be estimated from the data. Default is False. |
| box_cox_biasadj | bool | False | Whether to apply bias adjustment when inverting the Box-Cox transformation on forecasts. Default is False. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names. If provided, these columns will be treated as categorical variables and encoded accordingly. Default is None (no categorical variables). |
| categorical_encoder | Optional[Any] | None | Categorical encoder object (e.g. OneHotEncoder(), MeanEncoder(), TargetEncoder(), etc.) to apply to the categorical variables specified in cat_variables. The encoder should have fit() and transform() methods that can be applied to the input DataFrame. Default is None (no categorical encoding). |
| target_encode | bool | False | Whether target encoding is used for categorical variables. Default is False. |
| include_constant | bool | False | Whether to estimate an intercept/drift constant term in the ARIMA model. Default is False. |
| maxiter | int | 200 | Maximum number of L-BFGS-B iterations during parameter estimation. Default is 200. |
| tol | float | 1e-07 | Convergence tolerance for L-BFGS-B optimization. Default is 1e-7. |
| **Returns** | **None** |  |  |

### peshbeen.models.arima.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the model to the training data by applying the specified data preparation steps and estimating the ARIMA parameters.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target and any feature columns. |
| **Returns** | **arima** | **Fitted arima model instance.** |

### peshbeen.models.arima.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Recursive multi-step forecast.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. |
| exog | Optional[pd.DataFrame] | None | Optional dataframe of future regressors. |
| **Returns** | **np.ndarray** |  | **Forecast values of length `H`.** |

### peshbeen.models.arima.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run cross-validation using time series splits.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | DataFrame containing the target and any feature columns. |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of periods in each test set. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size to move the test window forward in each split. |
| h_split_point | Optional[int] | None | Optional index to split the test set into two parts for separate evaluation (e.g. to evaluate short-term vs long-term performance). If None, no split is done. |
| **Returns** | **pd.DataFrame** |  | **DataFrame containing predictions and true values for each split. Overall performance metrics averaged across splits are stored in `self.cv_summary`.** |

### Auto ARIMA forecaster

In [5]:
#| echo: false
render_api(auto_arima.__init__)
render_api(auto_arima.fit)
render_api(auto_arima.forecast)
render_api(auto_arima.cross_validate)
render_api(auto_arima.top_models)
render_api(auto_arima.get_orders)

### peshbeen.models.auto_arima

```python
auto_arima(
    target_col: 'str',
    seasonal_length: 'Optional[int]' = None,
    d: 'Optional[int]' = None,
    D: 'Optional[int]' = None,
    max_p: 'int' = 5,
    max_q: 'int' = 5,
    max_P: 'int' = 2,
    max_Q: 'int' = 2,
    max_d: 'int' = 2,
    max_D: 'int' = 1,
    start_p: 'int' = 2,
    start_q: 'int' = 2,
    start_P: 'int' = 1,
    start_Q: 'int' = 1,
    max_order: 'Optional[int]' = None,
    stepwise: 'bool' = True,
    criterion: 'str' = 'aicc',
    stationarity_test: 'str' = 'kpss',
    seasonal_test: 'str' = 'seas',
    alpha: 'float' = 0.05,
    with_intercept: 'Optional[bool]' = None,
    lag_transform: 'Optional[list]' = None,
    trend: 'Optional[str]' = None,
    pol_degree: 'int' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[List[int]]' = None,
    box_cox: 'Union[bool, float, int]' = False,
    box_cox_biasadj: 'bool' = False,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    target_encode: 'bool' = False,
    maxiter: 'int' = 200,
    tol: 'float' = 1e-07,
    trace: 'bool' = False,
    **kwargs
)
```

Initialize auto_arima model selection.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| target_col | str |  | Name of the target column in the input DataFrame. |
| seasonal_length | Optional[int] | None | Seasonal period length (e.g. 7 for daily with weekly pattern, 12 for monthly with annual pattern).
Default is None (non-seasonal). |
| d | Optional[int] | None | Integration/differencing order. If None (default), determined automatically via repeated
stationarity testing (KPSS). |
| D | Optional[int] | None | Seasonal differencing order (typically 0 or 1). If None (default), determined automatically
upfront using a seasonal unit root / strength test ('seasonal_test') before model search. |
| max_p | int | 5 | Maximum non-seasonal AR order. |
| max_q | int | 5 | Maximum non-seasonal MA order. |
| max_P | int | 2 | Maximum seasonal AR order. |
| max_Q | int | 2 | Maximum seasonal MA order. |
| max_d | int | 2 | Maximum non-seasonal differencing order tested during stationarity check. |
| max_D | int | 1 | Maximum seasonal differencing order tested when D is None. |
| start_p | int | 2 | Initial p order for Hyndman-Khandakar stepwise search. |
| start_q | int | 2 | Initial q order for Hyndman-Khandakar stepwise search. |
| start_P | int | 1 | Initial P order for seasonal Hyndman-Khandakar stepwise search. |
| start_Q | int | 1 | Initial Q order for seasonal Hyndman-Khandakar stepwise search. |
| max_order | Optional[int] | None | Maximum sum of orders (p + q + P + Q). If None, no cumulative constraint is applied. |
| stepwise | bool | True | Whether to use the Hyndman-Khandakar stepwise search algorithm. If False, an exhaustive
grid search over all combinations within max orders is performed. |
| criterion | str | aicc | Information criterion to minimize ('aicc', 'aic', or 'bic'). |
| stationarity_test | str | kpss | Statistical test used to determine non-seasonal differencing order d ('kpss' or 'adf'). |
| seasonal_test | str | seas | Method used to determine seasonal differencing order D when D is None:
- 'seas' (default): Wang-Smith-Hyndman seasonal strength test (F_s > 0.64).
- 'ocsb': Osborn-Chui-Smith-Birchenhall seasonal unit root test (requires pmdarima). |
| alpha | float | 0.05 | Significance level for the stationarity / seasonal differencing tests. |
| with_intercept | Optional[bool] | None | Whether to include a constant / drift term. If None (default), constant is tested according
to Hyndman-Khandakar rules (included unless d + D >= 2, and evaluated in variations). |
| lag_transform | Optional[list] | None | List of lag transform functions for exogenous feature generation. |
| trend | Optional[str] | None | Deterministic trend removal strategy ('linear' or 'ets'). |
| pol_degree | int | 1 | Degree of polynomial trend when trend='linear'. |
| ets_params | Optional[Dict[str, Any]] | None | Parameters for ETS trend model when trend='ets'. |
| change_points | Optional[List[int]] | None | Change point indices for piecewise linear trend. |
| box_cox | Union[bool, float, int] | False | Whether to apply Box-Cox transformation (or fixed lambda value). |
| box_cox_biasadj | bool | False | Whether to apply bias adjustment when back-transforming Box-Cox forecasts. |
| cat_variables | Optional[List[str]] | None | List of categorical covariate column names. |
| categorical_encoder | Optional[Any] | None | Categorical encoder instance (e.g. OneHotEncoder). |
| target_encode | bool | False | Whether target encoding is applied. |
| maxiter | int | 200 | Maximum L-BFGS-B iterations per candidate ARIMA model. |
| tol | float | 1e-07 | Convergence tolerance for optimizer. |
| trace | bool | False | Whether to print candidate models and their criterion scores during search. |
| kwargs | VAR_KEYWORD |  |  |
| **Returns** | **None** |  |  |

### peshbeen.models.auto_arima.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit auto_arima on the input time series DataFrame.

Conducts repeated stationarity tests to determine d, initializes the candidate models,
and executes stepwise or grid search to select the model minimizing the criterion.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Input DataFrame containing target_col and any exogenous regressors. |
| **Returns** | **'auto_arima'** | **Fitted auto_arima model instance.** |

### peshbeen.models.auto_arima.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Generate recursive multi-step out-of-sample forecasts using the best selected model.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon length. |
| exog | Optional[pd.DataFrame] | None | Future exogenous features of length H. Required if exogenous regressors were present during training. |
| **Returns** | **np.ndarray** |  | **1D array of length H containing point forecasts.** |

... (2 outputs left out)

### peshbeen.models.auto_arima.top_models

```python
top_models(
    n: 'int' = 5
)
```

Return the top n candidate models evaluated during the search ordered by the optimization criterion.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| n | int | 5 | Number of top models to report. |
| **Returns** | **pd.DataFrame** |  | **DataFrame containing model specifications, constant indicator, and criterion scores.** |

### peshbeen.models.auto_arima.get_orders

```python
get_orders(
    rank: 'int' = 1,
    top_n: 'Optional[int]' = None,
    include_score: 'bool' = False
)
```

Return model orders ('order', 'seasonal_order', 'seasonal_length', 'include_constant')
in a dictionary for the best model (default rank=1) or any runner-up model (rank=2, 3, ...),
or a list of dictionaries for the top n models.

The returned dictionary keys match the constructor arguments of ``arima`` so it can be
unpacked directly: ``arima(target_col=..., **model.get_orders(rank=2))``.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| rank | int | 1 | 1-based rank of the candidate model (1 = best model, 2 = second best, etc.). |
| top_n | Optional[int] | None | If provided, returns a list of parameter dictionaries for the top n models instead of a single dictionary. |
| include_score | bool | False | Whether to also include criterion scores ('criterion', 'score', 'aic', 'aicc', 'bic')
in the dictionary. Set to False (default) to unpack directly into ``arima``. |
| **Returns** | **Union[Dict[str, Any], List[Dict[str, Any]]]** |  | **Dictionary or list of dictionaries containing:
- 'order': tuple of int (p, d, q)
- 'seasonal_order': tuple of int (P, D, Q)
- 'seasonal_length': int or None
- 'include_constant': bool** |

### Machine learning forecaster

In [6]:
#| echo: false
render_api(ml_forecaster.__init__)
render_api(ml_forecaster.fit)
render_api(ml_forecaster.forecast)
render_api(ml_forecaster.cross_validate)

### peshbeen.models.ml_forecaster

```python
ml_forecaster(
    model: 'Any',
    target_col: 'str',
    lags: 'Optional[Union[int, List[int]]]' = None,
    lag_transform: 'Optional[list]' = None,
    difference: 'Optional[int]' = None,
    seasonal_diff: 'Optional[int]' = None,
    trend: 'Optional[str]' = None,
    pol_degree: 'int' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[List[int]]' = None,
    box_cox: 'Union[bool, float, int]' = False,
    box_cox_biasadj: 'bool' = False,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None
)
```

Initialize the ml_forecaster with the specified model and preprocessing options.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Any |  | A regression model object (e.g. LGBMRegressor(), XGBRegressor(), CatBoostRegressor(), LinearRegression(), etc.) |
| target_col | str |  | Name of the target variable column in the input DataFrame. |
| lags | Optional[Union[int, List[int]]] | None | Lags to include as features. If an integer is provided, lags from 1 to that integer will be included. If a list of integers is provided, those specific lags will be included. Default is None (no lag features). |
| lag_transform | Optional[list] | None | List of lag-transform function objects to apply to the target variable (e.g. [expanding_mean(shift=1), rolling_std(window_size=3, shift=1)]). Each function should take a pandas Series as input and return a Series of the same length. Default is None (no lag transforms). |
| difference | Optional[int] | None | Order of ordinary differencing to apply to the target variable (e.g. 1 for first difference). Default is None (no differencing). |
| seasonal_diff | Optional[int] | None | Seasonal period for seasonal differencing (e.g. 12 for monthly data with yearly seasonality). Default is None (no seasonal differencing). |
| trend | Optional[str] | None | Trend strategy to use. Options are 'linear' for linear trend removal, 'ets' for ETS-based trend removal, 'feature_lr' for using linear trend components as features, and 'feature_ets' for using ETS trend components as features. Default is None (no trend handling). |
| pol_degree | int | 1 | Degree of polynomial trend to fit when using 'linear' or 'feature_lr' trend strategy. Default is 1 (linear trend). |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary of parameters for the ExponentialSmoothing model when using 'ets' trend strategy. The keys should be the parameter names and the values should be the parameter values. Default is None (use default ETS parameters). |
| change_points | Optional[List[int]] | None | List of indices in the time series where change points occur for piecewise linear trend fitting. Only used when trend strategy is 'linear' or 'feature_lr'. Default is None (no change points, fit a single linear trend). |
| box_cox | Union[bool, float, int] | False | Whether to apply Box-Cox transformation to the target variable. If a float or int value is provided, it will be used as the lambda parameter for the Box-Cox transformation. If True, the lambda parameter will be estimated from the data. |
| box_cox_biasadj | bool | False | Whether to apply bias adjustment when inverting the Box-Cox transformation on forecasts. Default is False. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names. If provided, these columns will be treated as categorical variables and encoded accordingly. When categorical_encoder is None, native categorical support is used (e.g. LGBMRegressor, CatBoostRegressor, XGBRegressor with enable_categorical=True, or HistGradientBoostingRegressor with categorical_features='from_dtype'). Default is None (no categorical variables). |
| categorical_encoder | Optional[Any] | None | Categorical encoder object (e.g. OneHotEncoder(), MeanEncoder(), TargetEncoder(), etc.) to apply to the categorical variables specified in cat_variables. The encoder should have fit() and transform() methods that can be applied to the input DataFrame. Default is None (no categorical encoding) and if None, categorical variables can only be used if the model can handle them natively (e.g. LGBMRegressor, CatBoostRegressor, XGBRegressor, or HistGradientBoostingRegressor). |
| **Returns** | **None** |  |  |

### peshbeen.models.ml_forecaster.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the model to the training data after applying the specified data preparation steps.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target and any feature columns. |
| **Returns** | **None** |  |

### peshbeen.models.ml_forecaster.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Recursive multi-step forecast.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. |
| exog | Optional[pd.DataFrame] | None | Optional dataframe of future regressors. |
| **Returns** | **np.ndarray** |  | **Forecast values of length `H`.** |

### peshbeen.models.ml_forecaster.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run cross-validation using time series splits.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | DataFrame containing the target and any feature columns. |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of periods in each test set. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size to move the test window forward in each split. |
| h_split_point | Optional[int] | None | Optional index to split the test set into two parts for separate evaluation (e.g. to evaluate short-term vs long-term performance). If None, no split is done. |
| **Returns** | **Tuple[pd.DataFrame, pd.DataFrame]** |  | **DataFrame containing overall performance metrics averaged across splits, and a DataFrame with predictions and true values for each split.** |

### Direct machine learning forecaster

In [7]:
#| echo: false
render_api(ml_direct_forecaster.__init__)
render_api(ml_direct_forecaster.fit)
render_api(ml_direct_forecaster.forecast)
render_api(ml_direct_forecaster.cross_validate)

### peshbeen.models.ml_direct_forecaster

```python
ml_direct_forecaster(
    model: 'Any',
    target_col: 'str',
    H: 'Optional[Union[int, List[int]]]',
    lags: 'Optional[Union[int, List[int]]]' = None,
    lag_transform: 'Optional[list]' = None,
    difference: 'Optional[int]' = None,
    seasonal_diff: 'Optional[int]' = None,
    trend: 'Optional[str]' = None,
    pol_degree: 'int' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[List[int]]' = None,
    box_cox: 'Union[bool, float, int]' = False,
    box_cox_biasadj: 'bool' = False,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    future_exog_cols: 'Optional[List[str]]' = None
)
```

Initialize the ml_direct_forecaster with the specified model and preprocessing options.
Unlike ml_forecaster, this class uses a direct forecasting strategy: a separate model
is trained for each horizon h.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Any |  | A regression model object (e.g. LGBMRegressor(), XGBRegressor(), LinearRegression(), etc.) |
| target_col | str |  | Name of the target variable column in the input DataFrame. |
| H | Optional[Union[int, List[int]]] |  | Forecast horizon(s). If int, forecasts 1..H. If list, forecasts specified horizons. |
| lags | Optional[Union[int, List[int]]] | None | Lags to include as features. Default is None. |
| lag_transform | Optional[list] | None | Lag-transform functions to apply to the target variable. Default is None. |
| difference | Optional[int] | None | Order of ordinary differencing. Default is None. |
| seasonal_diff | Optional[int] | None | Seasonal period for seasonal differencing. Default is None. |
| trend | Optional[str] | None | Trend strategy: 'linear' or 'ets'. Default is None. |
| pol_degree | int | 1 | Polynomial degree for linear trend. Default is 1. |
| ets_params | Optional[Dict[str, Any]] | None | Parameters for ExponentialSmoothing when trend='ets'. Default is None. |
| change_points | Optional[List[int]] | None | Breakpoint indices for piecewise linear trend. Default is None. |
| box_cox | Union[bool, float, int] | False | Box-Cox transformation. If float/int, used as lambda. If True, lambda is estimated. Default is False. |
| box_cox_biasadj | bool | False | Bias adjustment when inverting Box-Cox. Default is False. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names. If provided, these columns will be treated as categorical variables and encoded accordingly. Default is None (no categorical variables). |
| categorical_encoder | Optional[Any] | None | Categorical encoder object (e.g. OneHotEncoder(), MeanEncoder(), etc.) to apply to the categorical variables specified in cat_variables. The encoder should have fit() and transform() methods that can be applied to the input DataFrame. Default is None (no categorical encoding) and if None, categorical variables can only be used if the model can handle them natively (e.g. LGBM or CatBoost). |
| future_exog_cols | Optional[List[str]] | None | List of column names for future exogenous variables that needs to be considered for each horizons. Those columns will be considered when exogenous variables, including those future values, are provided during forecasting. |
| **Returns** | **None** |  |  |

### peshbeen.models.ml_direct_forecaster.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit a separate model for each horizon h in 1..H.
For each h, the target is shifted h steps forward so the model learns
to predict the value h steps ahead directly from the current lag features,
bypassing recursive error accumulation.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target and any feature columns. |
| **Returns** | **None** |  |

### peshbeen.models.ml_direct_forecaster.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Generate direct multi-step forecasts. Each horizon h is predicted
independently by its own model using the most recent lag features —
no predictions are fed back as inputs.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. No need to specify any horizons. |
| exog | Optional[pd.DataFrame] | None | Optional future exogenous variables for the forecast period. Must contain the columns specified in future_exog_cols during initialization, and should be aligned with the forecast horizon (i.e. row 0 corresponds to t+1, row 1 to t+2, etc.). If the model was not initialized with future_exog_cols, providing exog will raise an error. |
| **Returns** | **np.ndarray** |  | **Forecast values of length H.** |

### peshbeen.models.ml_direct_forecaster.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run cross-validation using time series splits.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | DataFrame containing the target and any feature columns. |
| cv_split | int |  | Number of cross-validation splits. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size to move the test window forward in each split. |
| h_split_point | Optional[int] | None | Optional index to split the test set into two parts for separate evaluation (e.g. to evaluate short-term vs long-term performance). If None, no split is done. |
| **Returns** | **Tuple[pd.DataFrame, pd.DataFrame]** |  | **DataFrame containing overall performance metrics averaged across splits, and a DataFrame with predictions and true values for each split.** |

### GLM (Generalized Linear Model) forecaster

In [8]:
#| echo: false
render_api(glm.__init__)
render_api(glm.fit)
render_api(glm.forecast)
render_api(glm.cross_validate)

### peshbeen.models.glm

```python
glm(
    family: 'Any',
    target_col: 'str',
    lags: 'Optional[Union[int, List[int]]]' = None,
    lag_transform: 'Optional[list]' = None,
    difference: 'Optional[int]' = None,
    seasonal_diff: 'Optional[int]' = None,
    trend: 'Optional[str]' = None,
    pol_degree: 'int' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[List[int]]' = None,
    box_cox: 'Union[bool, float, int]' = False,
    box_cox_biasadj: 'bool' = False,
    add_constant: 'bool' = True,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    offset: 'Optional[np.ndarray]' = None,
    exposure: 'Optional[np.ndarray]' = None,
    freq_weights: 'Optional[np.ndarray]' = None,
    var_weights: 'Optional[np.ndarray]' = None,
    missing: 'Optional[str]' = None
)
```

Initialize the glm forecaster with the specified model and data preparation parameters.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| family | Any |  | A statsmodels family object specifying the error distribution and link function for the GLM (e.g. family=sm.families.Poisson() for count data, family=sm.families.Binomial() for binary data, etc.). `import statsmodels.api as sm`, so you can access the families via `sm.families`. |
| target_col | str |  | Name of the target variable column in the input DataFrame. |
| lags | Optional[Union[int, List[int]]] | None | Lags to include as features. If an integer is provided, lags from 1 to that integer will be included. If a list of integers is provided, those specific lags will be included. Default is None (no lag features). |
| lag_transform | Optional[list] | None | List of lag-transform function objects to apply to the target variable (e.g. [expanding_mean(shift=1), rolling_std(window_size=3, shift=1)]). Each function should take a pandas Series as input and return a Series of the same length. Default is None (no lag transforms). |
| difference | Optional[int] | None | Order of ordinary differencing to apply to the target variable (e.g. 1 for first difference). Default is None (no differencing). |
| seasonal_diff | Optional[int] | None | Seasonal period for seasonal differencing (e.g. 12 for monthly data with yearly seasonality). Default is None (no seasonal differencing). |
| trend | Optional[str] | None | Trend strategy to use. Options are 'linear' for linear trend removal, 'ets' for ETS-based trend removal, 'feature_lr' for using linear trend components as features, and 'feature_ets' for using ETS trend components as features. Default is None (no trend handling). |
| pol_degree | int | 1 | Degree of polynomial trend to fit when using 'linear' or 'feature_lr' trend strategy. Default is 1 (linear trend). |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary of parameters for the ExponentialSmoothing model when using 'ets' trend strategy. The keys should be the parameter names and the values should be the parameter values. Default is None (use default ETS parameters). |
| change_points | Optional[List[int]] | None | List of indices in the time series where change points occur for piecewise linear trend fitting. Only used when trend strategy is 'linear' or 'feature_lr'. Default is None (no change points, fit a single linear trend). |
| box_cox | Union[bool, float, int] | False | Whether to apply Box-Cox transformation to the target variable. If a float or int value is provided, it will be used as the lambda parameter for the Box-Cox transformation. If True, the lambda parameter will be estimated from the data. |
| box_cox_biasadj | bool | False | Whether to apply bias adjustment when inverting the Box-Cox transformation on forecasts. Default is False. |
| add_constant | bool | True |  |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names. If provided, these columns will be treated as categorical variables and encoded accordingly. Default is None (no categorical variables). |
| categorical_encoder | Optional[Any] | None | Categorical encoder object (e.g. OneHotEncoder(), MeanEncoder(), etc.) to apply to the categorical variables specified in cat_variables. The encoder should have fit() and transform() methods that can be applied to the input DataFrame. Default is None (no categorical encoding) and if None, categorical variables can only be used if the model can handle them natively (e.g. LGBM or CatBoost). |
| offset | Optional[np.ndarray] | None | An offset to be included in the model. If provided, must be an array whose length is the number of rows in exog. |
| exposure | Optional[np.ndarray] | None | Log(exposure) will be added to the linear prediction in the model. Exposure is only valid if the log link is used. If provided, it must be an array with the same length as endog. |
| freq_weights | Optional[np.ndarray] | None | 1d array of frequency weights. The default is None. If None is selected or a blank value, then the algorithm will replace with an array of 1’s with length equal to the endog. WARNING: Using weights is not verified yet for all possible options and results, see Notes in statsmodels documentation. |
| var_weights | Optional[np.ndarray] | None | 1d array of variance (analytic) weights. The default is None. If None is selected or a blank value, then the algorithm will replace with an array of 1’s with length equal to the endog. WARNING: Using weights is not verified yet for all possible options and results, see Notes in statsmodels documentation. |
| missing | Optional[str] | None | Available options are ‘none’, ‘drop’, and ‘raise’. If ‘none’, no nan checking is done. If ‘drop’, any observations with nans are dropped. If ‘raise’, an error is raised. Default is ‘none’. |
| **Returns** | **None** |  |  |

### peshbeen.models.glm.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the model to the training data after applying the specified data preparation steps.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target and any feature columns. |
| **Returns** | **None** |  |

### peshbeen.models.glm.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Recursive multi-step forecast.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. |
| exog | Optional[pd.DataFrame] | None | Optional dataframe of future regressors. |
| **Returns** | **np.ndarray** |  | **Forecast values of length `H`.** |

### peshbeen.models.glm.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run cross-validation using time series splits.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | DataFrame containing the target and any feature columns. |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of periods in each test set. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size to move the test window forward in each split. |
| h_split_point | Optional[int] | None | Optional index to split the test set into two parts for separate evaluation (e.g. to evaluate short-term vs long-term performance). If None, no split is done. |
| **Returns** | **Tuple[pd.DataFrame, pd.DataFrame]** |  | **DataFrame containing overall performance metrics averaged across splits, and a DataFrame with predictions and true values for each split.** |

### MS-ARR (Markov Switching Autoregressive Regression) forecaster

In [9]:
#| echo: false
render_api(ms_arr.__init__)
render_api(ms_arr.fit)
render_api(ms_arr.forecast)
render_api(ms_arr.cross_validate)

### peshbeen.models.ms_arr

```python
ms_arr(
    n_components: 'int',
    target_col: 'str',
    lags: 'Optional[Union[int, List[int]]]' = None,
    lag_transform: 'Optional[list]' = None,
    difference: 'Optional[int]' = None,
    seasonal_diff: 'Optional[int]' = None,
    trend: 'Optional[str]' = None,
    pol_degree: 'int' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[List[int]]' = None,
    box_cox: 'Union[bool, float, int]' = False,
    box_cox_biasadj: 'bool' = False,
    add_constant: 'bool' = True,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    method: 'str' = 'posterior',
    switching_var: 'bool' = True,
    startprob_prior: 'float' = 1000.0,
    transmat_prior: 'float' = 1000.0,
    n_iter: 'int' = 100,
    tol: 'float' = 0.001,
    ridge: 'float' = 1e-05,
    coefficients: 'Optional[np.ndarray]' = None,
    stds: 'Optional[np.ndarray]' = None,
    init_state: 'Optional[np.ndarray]' = None,
    trans_matrix: 'Optional[np.ndarray]' = None,
    random_state: 'int' = 42,
    verbose: 'bool' = False
)
```

Initialize the MS-ARR model with the specified parameters.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| n_components | int |  | Number of hidden states (regimes). |
| target_col | str |  | Name of the target variable. |
| lags | Optional[Union[int, List[int]]] | None | Lags for the autoregressive model. |
| lag_transform | Optional[list] | None | List of lag-transform function objects applied to the target. |
| difference | Optional[int] | None | Order of ordinary differencing (e.g. 1 for first difference). |
| seasonal_diff | Optional[int] | None | Seasonal period for seasonal differencing. |
| trend | Optional[str] | None | Trend strategy: 'linear' or 'ets'. |
| pol_degree | int | 1 | Degree of polynomial trend (default: 1). Used when trend='linear'. |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary of parameters for the ExponentialSmoothing model when using 'ets' trend strategy. The keys should be the parameter names and the values should be the parameter values. Default is None (use default ETS parameters). |
| change_points | Optional[List[int]] | None | Change points for piecewise linear trend. List of indices where the trend slope can change. |
| box_cox | Union[bool, float, int] | False | Whether to apply Box-Cox transformation to the target variable. If a float or int value is provided, it will be used as the lambda parameter for the Box-Cox transformation. If True, the lambda parameter will be estimated from the data. |
| box_cox_biasadj | bool | False | Whether to apply bias adjustment when inverting Box-Cox transformation (default: False). |
| add_constant | bool | True | If True, prepend a constant column to the regressor matrix (default: True). |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names. If provided, these columns will be treated as categorical variables and encoded accordingly. Default is None (no categorical variables). |
| categorical_encoder | Optional[Any] | None | Categorical encoder object (e.g. OneHotEncoder(), MeanEncoder(), etc.) to apply to the categorical variables specified in cat_variables. The encoder should have fit() and transform() methods that can be applied to the input DataFrame. Default is None (no categorical encoding) and if None, categorical variables can only be used if the model can handle them natively (e.g. LGBM or CatBoost). |
| method | str | posterior | State assignment method: 'posterior' (soft) or 'viterbi' (hard). Default: 'posterior'. |
| switching_var | bool | True | If True, each regime has its own variance. If False, uses pooled variance. Default: True. |
| startprob_prior | float | 1000.0 | Dirichlet concentration for initial state distribution. Default: 1e3. |
| transmat_prior | float | 1000.0 | Dirichlet concentration for transition matrix rows. Default: 1e3. |
| n_iter | int | 100 | Maximum EM iterations. Default: 100. |
| tol | float | 0.001 | Convergence tolerance on log-likelihood. Default: 1e-6. |
| ridge | float | 1e-05 | Ridge regularisation parameter for coefficient estimation. Default: 1e-5. |
| coefficients | Optional[np.ndarray] | None | Initial regression coefficients (shape: n_states x n_features). |
| stds | Optional[np.ndarray] | None | Initial state standard deviations (shape: n_states,). |
| init_state | Optional[np.ndarray] | None | Initial state probability vector (shape: n_states,). |
| trans_matrix | Optional[np.ndarray] | None | Initial transition matrix (shape: n_states x n_states). |
| random_state | int | 42 | Random seed for reproducibility. Default: 42. |
| verbose | bool | False | If True, print EM progress. Default: False. |
| **Returns** | **None** |  |  |

### peshbeen.models.ms_arr.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the model using the EM algorithm.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target and any feature columns. |
| **Returns** | **float** | **Final log-likelihood after EM convergence.** |

### peshbeen.models.ms_arr.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Generate forecasts for H future time steps.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon (number of steps to forecast ahead). |
| exog | Optional[pd.DataFrame] | None | Future exogenous regressors (must contain at least H rows). Should have the same columns as the training data (excluding the target variable). |
| **Returns** | **np.ndarray** |  |  |

### peshbeen.models.ms_arr.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    n_iter: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Run cross-validation.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | DataFrame containing the target and any feature columns. |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of time steps in the test set for each split. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | Step size between the start of each test set in the splits. |
| n_iter | int | 1 | Number of EM iterations to run for each training fold. |
| h_split_point | Optional[int] | None | If provided, split the test set into two parts at this index and evaluate metrics separately on each part. |
| **Returns** | **Union[pd.DataFrame, Tuple[pd.DataFrame, pd.DataFrame]]** |  | **DataFrame containing the average score for each metric across all splits. If h_split_point is provided, also includes separate scores for the two parts of the test set. If cv_df is True, returns a tuple of (overall_performance_df, cv_predictions_df)** |

## Multivariate forecasters

Multivariate forecasters model joint, interdependent dynamical systems where multiple target variables interact simultaneously at each time step (distinct from multi-series panel forecasters).

### VAR (Vector Autoregression) forecaster

In [10]:
#| echo: false
render_api(var.__init__)
render_api(var.fit)
render_api(var.forecast)
render_api(var.cross_validate)

### peshbeen.models.var

```python
var(
    target_cols: 'List[str]',
    lags: 'Dict[str, Union[int, List[int]]]',
    lag_transform: 'Optional[Dict[str, list]]' = None,
    difference: 'Optional[Dict[str, int]]' = None,
    seasonal_diff: 'Optional[Dict[str, int]]' = None,
    trend: 'Optional[Dict[str, str]]' = None,
    pol_degree: 'Optional[Union[int, Dict[str, int]]]' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[Dict[str, List[int]]]' = None,
    box_cox: 'Optional[Dict[str, Union[bool, float, int]]]' = None,
    box_cox_biasadj: 'Union[bool, Dict[str, bool]]' = False,
    add_constant: 'bool' = True,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Union[Dict[str, Any], Any]]' = None,
    verbose: 'bool' = False
)
```

"
Initialize the VAR model with specified preprocessing and modeling parameters.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| target_cols | List[str] |  | List of target column names to model. |
| lags | Dict[str, Union[int, List[int]]] |  | Dictionary specifying lags for each target variable. Values can be an int (number of lags) or a list of specific lag indices. |
| lag_transform | Optional[Dict[str, list]] | None | Dictionary specifying lag-transform functions for each target variable. Each value is a list of transformation functions (e.g., rolling_mean, expanding_std) to apply to the lagged features of that target. |
| difference | Optional[Dict[str, int]] | None | Dictionary specifying the order of ordinary differencing to apply to each target variable. Values are integers indicating how many times to difference the series. |
| seasonal_diff | Optional[Dict[str, int]] | None | Dictionary specifying the seasonal period for seasonal differencing for each target variable. Values are integers indicating the seasonal lag (e.g., 12 for monthly data with yearly seasonality). |
| trend | Optional[Dict[str, str]] | None | Dictionary specifying the trend strategy for each target variable. Values can be 'linear' for linear trend removal or 'ets' for ETS-based trend removal. |
| pol_degree | Optional[Union[int, Dict[str, int]]] | 1 | Polynomial degree for linear trend removal. Can be a single integer applied to all targets or a dictionary specifying the degree for each target. |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary specifying ETS model and fit parameters for each target variable when using 'ets' trend removal. Each value is a dictionary of parameters for the ExponentialSmoothing model and fitting process. |
| change_points | Optional[Dict[str, List[int]]] | None | Dictionary specifying change points for piecewise linear trend removal for each target variable. Values are lists of integer indices indicating where the trend should change. Only applicable when trend strategy is 'linear'. |
| box_cox | Optional[Dict[str, Union[bool, float, int]]] | None | Dictionary specifying whether to apply Box-Cox transformation to each target variable. Values can be a boolean (True to apply, False to skip) or a float (lambda parameter for Box-Cox transformation). If True, lambda will be estimated from the data. |
| box_cox_biasadj | Union[bool, Dict[str, bool]] | False | Whether to apply bias adjustment when inverting the Box-Cox transformation on forecasts. Can be a single boolean applied to all targets or a dictionary specifying the bias adjustment for each target. |
| add_constant | bool | True | If True, a constant column will be added to the regressor matrix for the VAR model. This is typically used to allow for an intercept in the model. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names to encode. These will be shared across all target variables. |
| categorical_encoder | Optional[Union[Dict[str, Any], Any]] | None | A categorical encoder instance, or a single-entry dictionary mapping the target column to the encoder when the encoder requires access to the target variable during fitting (e.g. {target_col: MeanEncoder()}). If encoder requiring target access is provided directly without the dict format, first target column in target_cols will be used for fitting the encoder. For encoders that do not require target access, pass the encoder instance directly (e.g. OneHotEncoder()). |
| verbose | bool | False | If True, the model will print verbose messages. |
| **Returns** | **None** |  |  |

### peshbeen.models.var.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the VAR model to the provided DataFrame.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing the target and any feature columns. |
| **Returns** | **None** |  |

### peshbeen.models.var.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Generate forecasts for H future time steps.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon (number of steps ahead to predict). |
| exog | Optional[pd.DataFrame] | None | Future exogenous regressors (must contain at least H rows). |
| **Returns** | **Dict[str, np.ndarray]** |  | **Forecasted values for each target, keyed by column name.** |

### peshbeen.models.var.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'Optional[int]' = None,
    h_split_point: 'Optional[int]' = None
)
```

Expanding-window cross-validation evaluated across all target variables.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | Input DataFrame containing all target variables and any feature columns. |
| cv_split | int |  | Number of cross-validation folds. |
| test_size | int |  | Test size (forecast horizon) per fold. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[WMAPE, MAE, RMSE, MASE]``) used to evaluate forecast accuracy across folds.
Access ``.cv_summary`` after cross-validation to retrieve the aggregated scores across all target variables. |
| step_size | Optional[int] | None | Step size for expanding window. If None, defaults to test_size. |
| h_split_point | Optional[int] | None | Point to split the test set for separate evaluation. Default is None. |
| **Returns** | **pd.DataFrame** |  | **DataFrame containing detailed cross-validation results with columns: fold, target, cutoff_date, horizon, y_true, y_pred.** |

### MS-VAR (Markov Switching Vector Autoregression) forecaster

In [11]:
#| echo: false
render_api(ms_var.__init__)
render_api(ms_var.fit)
render_api(ms_var.forecast)
render_api(ms_var.cross_validate)
render_api(ms_var.predict_states)
render_api(ms_var.predict_proba)

### peshbeen.models.ms_var

```python
ms_var(
    n_components: 'int',
    target_cols: 'List[str]',
    lags: 'Dict[str, Union[int, List[int]]]',
    lag_transform: 'Optional[Dict[str, list]]' = None,
    difference: 'Optional[Dict[str, int]]' = None,
    seasonal_diff: 'Optional[Dict[str, int]]' = None,
    trend: 'Optional[Dict[str, str]]' = None,
    pol_degree: 'Optional[Union[int, Dict[str, int]]]' = 1,
    ets_params: 'Optional[Dict[str, tuple]]' = None,
    change_points: 'Optional[Dict[str, List[int]]]' = None,
    box_cox: 'Optional[Dict[str, Union[bool, float, int]]]' = None,
    box_cox_biasadj: 'Optional[Dict[str, bool]]' = None,
    add_constant: 'bool' = True,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Union[Dict[str, Any], Any]]' = None,
    method: 'str' = 'posterior',
    covariance_type: 'str' = 'full',
    switching_cov: 'bool' = True,
    startprob_prior: 'float' = 1000.0,
    transmat_prior: 'float' = 100000.0,
    n_iter: 'int' = 100,
    tol: 'float' = 0.001,
    coefficients: 'Optional[List[np.ndarray]]' = None,
    init_state: 'Optional[np.ndarray]' = None,
    trans_matrix: 'Optional[np.ndarray]' = None,
    random_state: 'Optional[int]' = None,
    verbose: 'bool' = False
)
```

Initialize the Markov-Switching Vector Autoregression (MS-VAR) model with specified parameters and preprocessing options.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| n_components | int |  | Number of hidden states (regimes) in the model. |
| target_cols | List[str] |  | List of column names corresponding to the target variables in the input DataFrame. |
| lags | Dict[str, Union[int, List[int]]] |  | Dictionary specifying the lags to include for each target variable. The keys are target column names, and the values can be either an integer (which expands to a list of lags from 1 to that integer) or a list of specific lag integers. |
| lag_transform | Optional[Dict[str, list]] | None | Dictionary specifying lag-transform functions to apply to each target variable. The keys are target column names, and the values are lists of transformation functions (e.g., rolling mean, expanding std). |
| difference | Optional[Dict[str, int]] | None | Dictionary specifying the order of ordinary differencing to apply to each target variable. The keys are target column names, and the values are integers indicating the number of differences (e.g., 1 for first difference). |
| seasonal_diff | Optional[Dict[str, int]] | None | Dictionary specifying the order of seasonal differencing to apply to each target variable. The keys are target column names, and the values are integers indicating the seasonal period (e.g., 12 for monthly data with yearly seasonality). |
| trend | Optional[Dict[str, str]] | None | Dictionary specifying the trend component to include for each target variable. The keys are target column names, and the values are strings indicating the type of trend (e.g., 'linear', 'ets'). |
| pol_degree | Optional[Union[int, Dict[str, int]]] | 1 | Degree of polynomial for trend component. Can be a single integer applied to all targets or a dictionary keyed by target column name. |
| ets_params | Optional[Dict[str, tuple]] | None | Dictionary specifying ETS model parameters for trend removal. The keys are target column names, and the values are tuples containing the arguments for ExponentialSmoothing (first element) and its fit method (second element). |
| change_points | Optional[Dict[str, List[int]]] | None | Dictionary specifying change points for piecewise linear trend removal. The keys are target column names, and the values are lists of integer indices where the trend slope changes. |
| box_cox | Optional[Dict[str, Union[bool, float, int]]] | None | Dictionary specifying whether to apply Box-Cox transformation to each target variable. Values can be a boolean (True to apply, False to skip), a float (lambda parameter for Box-Cox transformation), or an integer (seasonal period for seasonal Box-Cox transformation). If True, lambda will be estimated from the data. |
| box_cox_biasadj | Optional[Dict[str, bool]] | None | Dictionary specifying whether to apply bias adjustment when inverting the Box-Cox transformation for each target variable. The keys are target column names, and the values are booleans indicating whether to apply bias adjustment. |
| add_constant | bool | True | If True, a constant column will be added to the regressor matrix for each state. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names to encode. These will be shared across all target variables. |
| categorical_encoder | Optional[Union[Dict[str, Any], Any]] | None | A categorical encoder instance, or a single-entry dictionary mapping the target column to the encoder when the encoder requires access to the target variable during fitting (e.g. {target_col: MeanEncoder()}). If encoder requiring target access is provided directly without the dict format, first target column in target_cols will be used for fitting the encoder. For encoders that do not require target access, pass the encoder instance directly (e.g. OneHotEncoder()). |
| method | str | posterior | Method for state assignment during the E-step. Options are 'posterior' for soft assignments based on posterior probabilities or 'viterbi' for hard assignments using the Viterbi algorithm. |
| covariance_type | str | full | Structure of the emission covariance matrices. Options are 'full' for full covariance matrices or 'diag' for diagonal covariance matrices. |
| switching_cov | bool | True | If True, allow the covariance matrices to switch between states. If False, a single covariance matrix will be shared across all states. |
| startprob_prior | float | 1000.0 | Concentration parameter for the Dirichlet prior on the initial state distribution. Higher values lead to more uniform priors. |
| transmat_prior | float | 100000.0 | Concentration parameter for the Dirichlet prior on the transition matrix rows. Higher values lead to more uniform priors. |
| n_iter | int | 100 | Maximum number of EM iterations to perform during training. |
| tol | float | 0.001 | Convergence threshold for EM. Training will stop if the change in log-likelihood between iterations is less than this value. |
| coefficients | Optional[List[np.ndarray]] | None | Optional initial state-wise VAR coefficient matrices. If None, they will be initialized using ordinary least squares on the entire dataset. |
| init_state | Optional[np.ndarray] | None | Optional initial state probability vector. If None, it will be initialized from a Dirichlet distribution with concentration parameter `startprob_prior`. |
| trans_matrix | Optional[np.ndarray] | None | Optional initial transition matrix. If None, it will be initialized from a Dirichlet distribution with concentration parameter `transmat_prior`. |
| random_state | Optional[int] | None | Random seed for reproducibility of the initial state distribution and transition matrix. |
| verbose | bool | False | If True, print log-likelihood at each EM iteration and convergence message. |
| **Returns** | **None** |  |  |

### peshbeen.models.ms_var.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the model using EM iterations until convergence on training data.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Training DataFrame containing all target and feature columns. Must include the columns specified in `target_cols` and any columns needed as regressors (e.g., categorical variables). |
| **Returns** | **float** | **Final log-likelihood after the additional EM iterations.** |

### peshbeen.models.ms_var.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Generate forecasts for H future time steps using the fitted MS-VAR model.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon (number of future time steps to predict). |
| exog | Optional[pd.DataFrame] | None | Future exogenous regressors. If provided, must contain at least H rows and the same columns as the regressors used during training (excluding target columns). |
| **Returns** | **Dict[str, np.ndarray]** |  | **Forecasted values for each target variable, returned as a dictionary keyed by target column name.** |

### peshbeen.models.ms_var.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'Optional[int]' = None,
    n_iter: 'int' = 1,
    h_split_point: 'Optional[int]' = None
)
```

Expanding-window cross-validation evaluated across all target variables.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | Input DataFrame containing all target variables and any feature columns. |
| cv_split | int |  | Number of cross-validation folds. |
| test_size | int |  | Test size (forecast horizon) per fold. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[WMAPE, MAE, RMSE, MASE]``) used to evaluate forecast accuracy across folds.
Access ``.cv_summary`` after cross-validation to retrieve the aggregated scores across all target variables. |
| step_size | Optional[int] | None | Step size for expanding window. If None, defaults to test_size. |
| n_iter | int | 1 | Number of EM iterations for each fold. Default is 1. |
| h_split_point | Optional[int] | None | Point to split the test set for separate evaluation. Default is None. |
| **Returns** | **pd.DataFrame** |  | **DataFrame containing detailed cross-validation results with columns: fold, target, cutoff_date, horizon, y_true, y_pred.** |

### peshbeen.models.ms_var.predict_states

```python
predict_states()
```

Return the most likely state at each time step (hard assignment).

### peshbeen.models.ms_var.predict_proba

```python
predict_proba()
```

Return the full posterior probability matrix (N x T).

## Multi-series forecasters

Multi-series forecasters train global models across collections of many individual time series sharing a common schema indexed by an entity ID (`id_col`). These should not be confused with multivariate forecasters (which model joint system dynamics among interacting variables).

### Panel machine learning forecaster

In [12]:
#| echo: false
render_api(panel_forecaster.__init__)
render_api(panel_forecaster.fit)
render_api(panel_forecaster.forecast)
render_api(panel_forecaster.cross_validate)

### peshbeen.models.panel_forecaster

```python
panel_forecaster(
    model: 'Any',
    id_col: 'str',
    target_col: 'str',
    lags: 'Optional[Union[int, List[int]]]' = None,
    lag_transform: 'Optional[List[Any]]' = None,
    id_col_encoder: 'Optional[Any]' = None,
    difference: 'Optional[Union[int, Dict[str, int]]]' = None,
    seasonal_diff: 'Optional[Union[int, Dict[str, int]]]' = None,
    trend: 'Optional[Union[str, Dict[str, str]]]' = None,
    pol_degree: 'Union[int, Dict[str, int]]' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[Union[list, Dict[str, list]]]' = None,
    box_cox: 'Optional[Union[bool, float, int, Dict[str, Any]]]' = False,
    box_cox_biasadj: 'Optional[Union[bool, Dict[str, bool]]]' = False,
    target_scaler: 'Optional[Any]' = None,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    **kwargs: 'Any'
)
```

Initialize the panel_forecaster with the specified model and preprocessing options.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Any |  | A regression model object (e.g. Ridge(), Lasso(), LinearRegression(), LGBMRegressor(), XGBRegressor(), CatBoostRegressor(), etc.). |
| id_col | str |  | Name of the column containing unique series identifiers. |
| target_col | str |  | Name of the target variable column in the input DataFrame. |
| lags | Optional[Union[int, List[int]]] | None | Lags to include as features across all series in the panel. If an integer N is provided, lags 1 to N (named lag_1, ..., lag_N) are created for each series and stacked vertically. If a list of integers is provided, those specific lags are created. Default is None (no lag features). |
| lag_transform | Optional[List[Any]] | None | List of lag-transformation function objects applied to each series independently and stacked vertically (e.g. [expanding_mean(shift=1), rolling_quantile(window_size=7, quantile=0.5, shift=1)]). Default is None (no lag transforms). |
| id_col_encoder | Optional[Any] | None | Strategy or scikit-learn transformer to encode unique series identifiers in panel data. Options are:
- None: Pass series ID directly as a native categorical feature to models with native categorical support (LGBMRegressor, CatBoostRegressor, XGBRegressor, and HistGradientBoostingRegressor).
- scikit-learn transformer: Any transformer instance such as OneHotEncoder(sparse_output=False), OrdinalEncoder(), TargetEncoder(), etc. |
| difference | Optional[Union[int, Dict[str, int]]] | None | Order of ordinary differencing to apply to each series before modeling. Default is None (no differencing). |
| seasonal_diff | Optional[Union[int, Dict[str, int]]] | None | Seasonal period for seasonal differencing (e.g. 7 for weekly, 12 for monthly, 24 for hourly). Default is None (no seasonal differencing). |
| trend | Optional[Union[str, Dict[str, str]]] | None | Trend removal strategy. Options are:
- 'linear': Global or piecewise linear trend estimation and removal.
- 'ets': Holt-Winters Exponential Smoothing trend estimation and removal.
Default is None (no trend removal). |
| pol_degree | Union[int, Dict[str, int]] | 1 | Degree of polynomial trend to fit when using 'linear' trend strategy. |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary of keyword arguments passed to statsmodels ExponentialSmoothing when trend='ets'. |
| change_points | Optional[Union[list, Dict[str, list]]] | None | List of integer time indices where slope change points occur for piecewise linear trend fitting. Default is None. |
| box_cox | Optional[Union[bool, float, int, Dict[str, Any]]] | False | Whether to apply Box-Cox transformation for variance stabilization. If True, estimates optimal lambda. If float/int, uses that value as fixed lambda. |
| box_cox_biasadj | Optional[Union[bool, Dict[str, bool]]] | False | Whether to apply bias adjustment when back-transforming Box-Cox forecasts. Default is False. |
| target_scaler | Optional[Any] | None | Scikit-learn compatible scaler instance (e.g. StandardScaler(), RobustScaler(), MinMaxScaler()) or dictionary of per-series scalers. Scalers are fitted on each series' target data and inverted on forecasts. Default is None. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names in exogenous data. When categorical_encoder is None, native categorical support is used (e.g. LGBMRegressor, CatBoostRegressor, XGBRegressor with enable_categorical=True, or HistGradientBoostingRegressor with categorical_features='from_dtype'). Default is None. |
| categorical_encoder | Optional[Any] | None | Scikit-learn compatible transformer (e.g. OneHotEncoder(drop='first', sparse_output=False), OrdinalEncoder(), TargetEncoder()) to encode `cat_variables`. If None, categorical features must be natively supported by the model. Default is None. |
| kwargs | Any |  |  |
| **Returns** | **None** |  |  |

### peshbeen.models.panel_forecaster.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the panel forecaster on the input long-format panel DataFrame.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Long-format panel DataFrame containing time index, id_col, and target_col. |
| **Returns** | **None** |  |

### peshbeen.models.panel_forecaster.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Recursive multi-step forecast across all panel series using per-series lag buffers.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon length. |
| exog | Optional[pd.DataFrame] | None | Future exogenous features DataFrame covering horizon H. Default is None. |
| **Returns** | **Dict[str, np.ndarray]** |  | **Dictionary keyed by series ID with length-H forecast arrays on original measurement scale.** |

### peshbeen.models.panel_forecaster.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'Optional[int]' = None,
    ref_series_id: 'Optional[str]' = None
)
```

Expanding-window cross-validation evaluated across all series in the panel.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | pd.DataFrame |  | Long-format panel DataFrame containing time index, id_col, and target_col. |
| cv_split | int |  | Number of cross-validation folds. |
| test_size | int |  | Number of time steps in the test set for each fold. |
| metrics | List[Callable] |  | List of metric functions to evaluate predictions (e.g., MAE, RMSE, MASE). |
| step_size | Optional[int] | None | Step size for expanding window. If None, defaults to test_size. |
| ref_series_id | Optional[str] | None | Reference series ID to determine cutoff dates for cross-validation. If None, the series with the fewest observations is used. |
| **Returns** | **pd.DataFrame** |  | **DataFrame containing detailed cross-validation results with columns: fold, id_col, cutoff_date, horizon, y_true, y_pred.** |

### Multi-series machine learning forecaster

In [13]:
#| echo: false
render_api(ml_multi_forecaster.__init__)
render_api(ml_multi_forecaster.fit)
render_api(ml_multi_forecaster.forecast)
render_api(ml_multi_forecaster.cross_validate)

### peshbeen.models.ml_multi_forecaster

```python
ml_multi_forecaster(
    model: 'Any',
    id_col: 'str',
    target_col: 'str',
    lags: 'Optional[Union[int, List[int], Dict[str, Union[int, List[int]]]]]' = None,
    lag_transform: 'Optional[Union[List[Any], Dict[str, List[Any]]]]' = None,
    id_col_encoder: 'Optional[Any]' = None,
    difference: 'Optional[Union[int, Dict[str, int]]]' = None,
    seasonal_diff: 'Optional[Union[int, Dict[str, int]]]' = None,
    trend: 'Optional[Union[str, Dict[str, str]]]' = None,
    pol_degree: 'Union[int, Dict[str, int]]' = 1,
    ets_params: 'Optional[Dict[str, Any]]' = None,
    change_points: 'Optional[Union[list, Dict[str, list]]]' = None,
    box_cox: 'Optional[Union[bool, float, int, Dict[str, Any]]]' = False,
    box_cox_biasadj: 'Optional[Union[bool, Dict[str, bool]]]' = False,
    target_scaler: 'Optional[Any]' = None,
    cat_variables: 'Optional[List[str]]' = None,
    categorical_encoder: 'Optional[Any]' = None,
    **kwargs: 'Any'
)
```

Initialize the ml_multi_forecaster with the specified model and preprocessing options.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Any |  | A regression model object (e.g. Ridge(), Lasso(), LinearRegression(), LGBMRegressor(), CatBoostRegressor(), etc.). |
| id_col | str |  | Name of the column containing unique series identifiers. |
| target_col | str |  | Name of the target variable column in the input DataFrame. |
| lags | Optional[Union[int, List[int], Dict[str, Union[int, List[int]]]]] | None | Lags to include as features. Can be specified globally as an integer (lags 1 to N) or list of integers, or as a dictionary mapping each series ID to its specific lag configuration. Default is None (no lag features). |
| lag_transform | Optional[Union[List[Any], Dict[str, List[Any]]]] | None | List of lag-transformation functions (e.g. [expanding_mean(shift=1), rolling_quantile(window_size=7, quantile=0.5, shift=1)]). Can be specified globally or per series as a dictionary. Default is None (no lag transforms). |
| id_col_encoder | Optional[Any] | None | Strategy or scikit-learn transformer to encode unique series identifiers in panel data. Options are:
- None: Pass series ID directly as a native categorical feature to models with native categorical support (LGBMRegressor, CatBoostRegressor, XGBRegressor, and HistGradientBoostingRegressor).
- scikit-learn transformer: Any transformer instance such as OneHotEncoder(sparse_output=False), OrdinalEncoder(), TargetEncoder(), etc.
- 'ordinal': Convenience shortcut for OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1). |
| difference | Optional[Union[int, Dict[str, int]]] | None | Order of ordinary differencing to apply to each series before modeling. Default is None (no differencing). |
| seasonal_diff | Optional[Union[int, Dict[str, int]]] | None | Seasonal period for seasonal differencing (e.g. 7 for weekly, 12 for monthly, 24 for hourly). Default is None (no seasonal differencing). |
| trend | Optional[Union[str, Dict[str, str]]] | None | Trend removal strategy. Options are:
- 'linear': Global or piecewise linear trend estimation and removal.
- 'ets': Holt-Winters Exponential Smoothing trend estimation and removal.
Default is None (no trend removal). |
| pol_degree | Union[int, Dict[str, int]] | 1 | Degree of polynomial trend to fit when using 'linear' trend strategy. |
| ets_params | Optional[Dict[str, Any]] | None | Dictionary of keyword arguments passed to statsmodels ExponentialSmoothing when trend='ets'. |
| change_points | Optional[Union[list, Dict[str, list]]] | None | List of integer time indices where slope change points occur for piecewise linear trend fitting. Default is None. |
| box_cox | Optional[Union[bool, float, int, Dict[str, Any]]] | False | Whether to apply Box-Cox transformation for variance stabilization. If True, estimates optimal lambda. If float/int, uses that value as fixed lambda. |
| box_cox_biasadj | Optional[Union[bool, Dict[str, bool]]] | False | Whether to apply bias adjustment when back-transforming Box-Cox forecasts. Default is False. |
| target_scaler | Optional[Any] | None | Scikit-learn compatible scaler instance (e.g. StandardScaler(), RobustScaler(), MinMaxScaler()) or dictionary of per-series scalers. Scalers are fitted on each series' target data and inverted on forecasts. Default is None. |
| cat_variables | Optional[List[str]] | None | List of categorical feature column names in exogenous data. When categorical_encoder is None, native categorical support is used (e.g. LGBMRegressor, CatBoostRegressor, XGBRegressor with enable_categorical=True, or HistGradientBoostingRegressor with categorical_features='from_dtype'). Default is None. |
| categorical_encoder | Optional[Any] | None | Scikit-learn compatible transformer (e.g. OneHotEncoder(drop='first', sparse_output=False), OrdinalEncoder(), TargetEncoder()) to encode `cat_variables`. If None, categorical features must be natively supported by the model. Default is None. |
| kwargs | Any |  |  |
| **Returns** | **None** |  |  |

### peshbeen.models.ml_multi_forecaster.fit

```python
fit(
    df: 'pd.DataFrame'
)
```

Fit the multi-series forecaster on the input panel DataFrame.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | pd.DataFrame | Long-format panel DataFrame containing time index, id_col, and target_col. |
| **Returns** | **None** |  |

### peshbeen.models.ml_multi_forecaster.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[pd.DataFrame]' = None
)
```

Recursive multi-step forecast across all target series using pre-allocated high-speed memory buffers.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon length. |
| exog | Optional[pd.DataFrame] | None | Future exogenous features DataFrame covering horizon H. Default is None. |
| **Returns** | **Dict[str, np.ndarray]** |  | **Dictionary keyed by series ID with length-H forecast arrays on original measurement scale.** |

### peshbeen.models.ml_multi_forecaster.cross_validate

```python
cross_validate(
    df: 'pd.DataFrame',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'Optional[int]' = None,
    ref_series_id: 'Optional[str]' = None
)
```

Expanding-window cross-validation evaluated across all series.

## Ensemble forecaster

### Hybrid (pesh) forecaster

In [14]:
#| echo: false
render_api(pesh.__init__)
render_api(pesh.fit)
render_api(pesh.forecast)
render_api(pesh.cross_validate)

### peshbeen.models.pesh

```python
pesh(
    models: 'dict',
    weighting_scheme: 'Optional[Dict[str, float]]' = None
)
```

Initialize the pesh model with the specified parameters for hybrid forecasting that combines forecasts from multiple models.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| models | dict |  | A dictionary of model instances to be used for forecasting. The keys should be string names for each model. |
| weighting_scheme | Optional[Dict[str, float]] | None | Optional dictionary specifying weights for each model's forecast. Default is None, which means equal weighting. |
| **Returns** | **None** |  |  |

### peshbeen.models.pesh.fit

```python
fit(
    df: 'Union[pd.DataFrame, dict]'
)
```

Fit the specified models to the training data.

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| df | Union[pd.DataFrame, dict] | Training DataFrame or dictionary containing the target and any feature columns specified for the models. If a dictionary is provided, it should have the same keys as the models dictionary, with each value being the corresponding DataFrame for that model. |
| **Returns** | **None** |  |

### peshbeen.models.pesh.forecast

```python
forecast(
    H: 'int',
    exog: 'Optional[Union[pd.DataFrame, dict]]' = None
)
```

Recursive multi-step forecast.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| H | int |  | Forecast horizon. |
| exog | Optional[Union[pd.DataFrame, dict]] | None | Optional dataframe or dictionary of dataframes containing exogenous variables for the forecast period. If a dictionary is provided, it should have the same keys as the models dictionary, with each value being the corresponding DataFrame for that model. |
| **Returns** | **pd.DataFrame** |  | **Forecast values for each model and the combined 'pesh' forecast.** |

### peshbeen.models.pesh.cross_validate

```python
cross_validate(
    df: 'Union[pd.DataFrame, dict]',
    cv_split: 'int',
    test_size: 'int',
    metrics: 'List[Callable]',
    step_size: 'int' = 1,
    metric_to_opt: 'Optional[Callable]' = None,
    weighting_scheme: 'Optional[Union[Dict[str, float], str]]' = None,
    optimizer: 'str' = 'SLSQP',
    h_split_point: 'Optional[int]' = None
)
```

Perform cross-validation for the pesh model using a rolling forecasting origin approach.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | Union[pd.DataFrame, dict] |  | The input DataFrame or dictionary containing the target and any feature columns. |
| cv_split | int |  | The number of cross-validation splits. |
| test_size | int |  | The size of the test set for each split. |
| metrics | List[Callable] |  | Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores. |
| step_size | int | 1 | The step size for rolling the forecasting origin. |
| metric_to_opt | Optional[Callable] | None | An optional metric function to optimize when weighting_scheme is set to "optimize". If None, it defaults to the first metric in the metrics list. |
| weighting_scheme | Optional[Union[Dict[str, float], str]] | None | None: equal weights across models. dict: user-provided weights (must sum to 1). "optimize": optimize weights to minimize MSE via `scipy.optimize.minimize`. |
| optimizer | str | SLSQP | Optimization method to use when weighting_scheme is set to "optimize". Passed to `scipy.optimize.minimize`. Refer to SciPy documentation for available methods. |
| h_split_point | Optional[int] | None | An optional integer to specify a split point for horizon-based performance evaluation. If provided, metrics will be calculated separately for forecasts with horizon <= h_split_point and > h_split_point, in addition to overall performance. |
| **Returns** | **pd.DataFrame** |  | **A DataFrame containing the performance metrics for each model and the combined forecast across all cross-validation splits. Also, optimized weights are stored in `self.optimal_weights_` if `weighting_scheme` is "optimize".** |

# Probabilistic forecasting

## Probabilistic forecasting for univariate time series

In [15]:
#| hide
from peshbeen.probabilistic_forecasting import prob_forecasts, mv_prob_forecasts, multi_prob_forecasts

In [16]:
#| echo: false
render_api(prob_forecasts.__init__)
render_api(prob_forecasts.calibrate)
render_api(prob_forecasts.sample)
render_api(prob_forecasts.sample_quantiles)
render_api(prob_forecasts.prediction_intervals)

### peshbeen.probabilistic_forecasting.prob_forecasts

```python
prob_forecasts(
    model,
    H: int,
    n_calibration: int | None = None,
    step_size: int = 1,
    random_state: int = 42,
    n_iter: int | None = None,
    verbose: bool = False
)
```



|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model |  |  | Any model with ``.target_col``, ``.fit(df)``, and ``.forecast(H, exog)`` attributes. |
| H | int |  | Forecast horizon. |
| n_calibration | Union | None | Number of calibration windows for cross-validated residual estimation. If None, in-sample residuals are used. |
| step_size | int | 1 | Step size between consecutive calibration windows. |
| random_state | int | 42 | Seed for internal random-number generators. |
| n_iter | Union | None | Number of EM iterations during calibration (relevant for ms_arr). |
| verbose | bool | False | Print progress during calibration. |

### peshbeen.probabilistic_forecasting.prob_forecasts.calibrate

```python
calibrate(
    df: pandas.core.frame.DataFrame,
    delta: float | List[float] = 0.5
)
```



### peshbeen.probabilistic_forecasting.prob_forecasts.sample

```python
sample(
    df: pandas.core.frame.DataFrame,
    n_samples: int = 1000,
    method: str = 'empirical',
    future_exog: pandas.core.frame.DataFrame | None = None,
    non_negative: bool = True
)
```

Draw sample paths from the predictive distribution.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | DataFrame |  | Training data. |
| n_samples | int | 1000 | Number of simulated trajectories. |
| method | str | empirical | - "empirical": Independent empirical residual resampling per horizon.
- "kde": Kernel Density Estimation smoothed sampling.
- "gaussian" or "mvn": Gaussian error sampling. If n_calibration is provided, samples from a multivariate normal fitted to cross-horizon residual covariance; if n_calibration is None, samples from a normal distribution using training residual variance.
- "copula": Gaussian Copula coupling cross-horizon correlation with empirical marginals. |
| future_exog | Union | None | Future exogenous features. |
| non_negative | bool | True | Enforce non-negative sample paths. |
| **Returns** | **prob_forecasts** |  |  |

### peshbeen.probabilistic_forecasting.prob_forecasts.sample_quantiles

```python
sample_quantiles(
    quantiles: float | List[float] = [0.05, 0.5, 0.95]
)
```



### peshbeen.probabilistic_forecasting.prob_forecasts.prediction_intervals

```python
prediction_intervals(
    coverage: float | List[float] = 0.9,
    conformal: bool = False,
    future_exog: pandas.core.frame.DataFrame | None = None
)
```



## Probabilistic forecasting for multivariate time series

In [17]:
#| echo: false
render_api(mv_prob_forecasts.__init__)
render_api(mv_prob_forecasts.calibrate)
render_api(mv_prob_forecasts.sample)
render_api(mv_prob_forecasts.sample_quantiles)
render_api(mv_prob_forecasts.prediction_intervals)

### peshbeen.probabilistic_forecasting.mv_prob_forecasts

```python
mv_prob_forecasts(
    model: Any,
    H: int,
    n_calibration: int | None = None,
    step_size: int = 1,
    random_state: int = 42,
    n_iter: int | None = None,
    verbose: bool = False
)
```

Initialize self.  See help(type(self)) for accurate signature.

### peshbeen.probabilistic_forecasting.mv_prob_forecasts.calibrate

```python
calibrate(
    df: pandas.core.frame.DataFrame,
    delta: float | List[float] = 0.5
)
```



### peshbeen.probabilistic_forecasting.mv_prob_forecasts.sample

```python
sample(
    df: pandas.core.frame.DataFrame,
    n_samples: int = 1000,
    method: str = 'empirical',
    future_exog: pandas.core.frame.DataFrame | None = None,
    target_series: str | List[str] | None = None,
    df_t_deg: float = 5.0,
    non_negative: bool = True
)
```

Draw sample paths from the predictive distribution across all target variables.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | DataFrame |  | Training data. |
| n_samples | int | 1000 | Number of simulated trajectories. |
| method | str | empirical | - "mvn" (or "gaussian"): Multivariate Normal on cross-series residual covariance.
- "student_t": Multivariate Student-t on cross-series residual covariance.
- "copula": Gaussian Copula coupling cross-series correlation with empirical marginals.
- "empirical": Joint empirical residual resampling across series (preserves cross-series covariance).
- "kde": Multivariate Kernel Density Estimation smoothed sampling preserving cross-series covariance.
- "ind_empirical" (or "ind_emp"): Independent empirical residual resampling per series and horizon.
- "ind_kde": Independent 1D Kernel Density Estimation smoothed sampling per series and horizon. |
| future_exog | Union | None | Future exogenous features. |
| target_series | Union | None | Specify which target variable(s) to return sample paths for. If None, returns all target variables. |
| df_t_deg | float | 5.0 | Degrees of freedom for the Student-t distribution (only used if method="student_t"). |
| non_negative | bool | True | Enforce non-negative sample paths. |
| **Returns** | **mv_prob_forecasts** |  | **A new instance of mv_prob_forecasts containing the simulated sample paths and point forecasts.** |

### peshbeen.probabilistic_forecasting.mv_prob_forecasts.sample_quantiles

```python
sample_quantiles(
    quantiles: float | List[float] = [0.05, 0.5, 0.95],
    target_series: str | List[str] | None = None
)
```

Compute quantiles from simulated sample paths across all or requested target variables.

### peshbeen.probabilistic_forecasting.mv_prob_forecasts.prediction_intervals

```python
prediction_intervals(
    coverage: float | List[float] = 0.9,
    conformal: bool = False,
    target_series: str | List[str] | None = None,
    future_exog: pandas.core.frame.DataFrame | None = None
)
```

Compute prediction intervals for all or requested target variables.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| coverage | Union | 0.9 | Desired nominal coverage level(s) (e.g. 0.90 for 90% prediction intervals). |
| conformal | bool | False | If False (default), intervals are computed from simulated sample paths.
If True, distribution-free conformal prediction intervals are computed using self.non_conform. |
| target_series | Union | None | Filter output to specific target variable(s). |
| future_exog | Union | None | Future exogenous variables if conformal=True and .sample() was not called first. |
| **Returns** | **Dict** |  |  |

## Probabilistic forecasting for multi-series panel models

Generates prediction intervals and scenarios for pooled multi-series models (`panel_forecaster`, `ml_multi_forecaster`).

In [18]:
#| echo: false
render_api(multi_prob_forecasts.__init__)
render_api(multi_prob_forecasts.calibrate)
render_api(multi_prob_forecasts.sample)
render_api(multi_prob_forecasts.sample_quantiles)
render_api(multi_prob_forecasts.prediction_intervals)

### peshbeen.probabilistic_forecasting.multi_prob_forecasts

```python
multi_prob_forecasts(
    model: Any,
    H: int,
    n_calibration: int | None = None,
    step_size: int = 1,
    ref_series_id: str | None = None,
    random_state: int = 42,
    verbose: bool = False
)
```

Initialize self.  See help(type(self)) for accurate signature.

### peshbeen.probabilistic_forecasting.multi_prob_forecasts.calibrate

```python
calibrate(
    df: pandas.core.frame.DataFrame,
    delta: float | List[float] = 0.5
)
```



### peshbeen.probabilistic_forecasting.multi_prob_forecasts.sample

```python
sample(
    df: pandas.core.frame.DataFrame,
    n_samples: int = 1000,
    method: str = 'empirical',
    future_exog: pandas.core.frame.DataFrame | None = None,
    target_series: str | List[str] | None = None,
    df_t_deg: float = 5.0,
    non_negative: bool = True
)
```

Draw sample paths from the predictive distribution.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| df | DataFrame |  |  |
| n_samples | int | 1000 |  |
| method | str | empirical |  |
| future_exog | Union | None |  |
| target_series | Union | None |  |
| df_t_deg | float | 5.0 |  |
| non_negative | bool | True |  |
| **Returns** | **multi_prob_forecasts** |  | **A new instance of multi_prob_forecasts containing the simulated sample paths and point forecasts.** |

### peshbeen.probabilistic_forecasting.multi_prob_forecasts.sample_quantiles

```python
sample_quantiles(
    quantiles: float | List[float] = [0.05, 0.5, 0.95],
    target_series: str | List[str] | None = None
)
```

Compute quantiles from simulated sample paths.

### peshbeen.probabilistic_forecasting.multi_prob_forecasts.prediction_intervals

```python
prediction_intervals(
    coverage: float | List[float] = 0.9,
    conformal: bool = False,
    target_series: str | List[str] | None = None,
    future_exog: pandas.core.frame.DataFrame | None = None
)
```

Compute prediction intervals for all or requested series.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| coverage | Union | 0.9 | Desired nominal coverage level(s) (e.g. 0.90 for 90% prediction intervals). |
| conformal | bool | False | If False (default), intervals are computed from simulated sample paths.
If True, distribution-free conformal prediction intervals are computed using self.non_conform. |
| target_series | Union | None | Filter output to specific series ID(s). |
| future_exog | Union | None | Future exogenous variables if conformal=True and .sample() was not called first. |
| **Returns** | **Dict** |  |  |

# Model selection and evaluation

## Hyperparameters tuning methods for Univariate machine learning models

In [19]:
#| hide
from peshbeen.model_selection import (
    optuna_tune,
    optuna_tune_multi,
    forward_feature_selection,
    backward_feature_selection
)

In [20]:
#| echo: false
render_api(optuna_tune)

### peshbeen.model_selection.optuna_tune

```python
optuna_tune(
    model: Any,
    df: pandas.core.frame.DataFrame,
    cv_split: int,
    test_size: int,
    eval_metric: Callable,
    param_space: Dict[str, Any],
    step_size: int = None,
    eval_num: int = 100,
    multivariate: bool = False,
    warm_up_steps: int = 10,
    startup_trials: int = 5,
    candidate_exog: List[str] = None,
    pareto_bounds: float | Tuple[float, float] = (0.5, 0.999),
    eval_horizons: None | int = None,
    verbose: bool = False,
    random_state: int | None = 42
)
```

Tune forecasting model hyperparameters using time series cross-validation and Optuna.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Any |  | Forecasting model with .fit and .forecast methods. |
| df | DataFrame |  | Time series data (datetime index, target column, optional exogenous features). |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of samples in each test fold. For ml_direct_forecaster, this will be overridden to be the maximum horizon in model.H. |
| eval_metric | Callable |  | Metric function to minimise. |
| param_space | Dict |  | Each value must be a callable that accepts an Optuna `trial` and returns a value. |
| step_size | int | None | Step size between CV folds. |
| eval_num | int | 100 | Number of Optuna trials. Default 100. |
| multivariate | bool | False | If True, uses TPE (Tree-structured Parzen Estimator) sampler for multivariate hyperparameter optimization. If False, uses default Independent sampler. |
| warm_up_steps | int | 10 | The number of initial CV folds to complete before the Optuna pruner begins evaluating trial performance. This "warm-up" period prevents the pruner from prematurely terminating promising trials due to high variance or noise present in the earliest (smallest) cross-validation folds. Default 10. |
| startup_trials | int | 5 | The number of complete trials (hyperparameter combinations) to evaluate before enabling the pruner. This ensures the pruner has a solid baseline median to compare against. Default 5. |
| candidate_exog | List | None | List of exogenous feature names to consider for feature importance-based selection. If None, no feature selection is performed. |
| pareto_bounds | Union | (0.5, 0.999) | If a float is provided, it is used as a fixed cutoff for cumulative importance (e.g., 0.8 means keep features that explain 80% of variance). If a tuple is provided, it defines the lower and upper bounds for tuning the Pareto cutoff. Default is (0.5, 0.999), meaning the cutoff will be tuned between 50% and 99.9% of cumulative importance. |
| eval_horizons | Union | None | If an integer is provided, the evaluation metric will be calculated starting from that horizon onward (e.g., if 5 is passed, the metric will be calculated on horizons 5, 6, 7, etc.). If None, the metric will be calculated on all horizons. |
| verbose | bool | False | Print score for every trial. Default False. |
| random_state | Union | 42 |  |
| **Returns** | **Tuple** |  | **Best hyperparameters and best lags (if 'lags' is in param_space).** |

## Hyperparameters tuning methods for Multivariate machine learning models

In [21]:
#| echo: false
render_api(optuna_tune_multi)

### peshbeen.model_selection.optuna_tune_multi

```python
optuna_tune_multi(
    model: Any,
    df: pandas.core.frame.DataFrame,
    cv_split: int,
    test_size: int,
    eval_metric: Callable,
    param_space: Dict[str, Any],
    target_series: str | List[str] | None = None,
    step_size: int = 1,
    eval_num: int = 100,
    multivariate: bool = False,
    warm_up_steps: int = 10,
    startup_trials: int = 5,
    ref_series_id: str | None = None,
    transform_back: bool = True,
    verbose: bool = False,
    random_state: int | None = 42
)
```

Tune multi-series panel (`ml_multi_forecaster`, `panel_forecaster`) or wide multivariate
(`ml_multi_output_forecaster`, `var`, `ms_var`) forecasting model hyperparameters
using time-series cross-validation and Optuna.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Any |  | An instance of `ml_multi_forecaster`, `panel_forecaster`, `ml_multi_output_forecaster`, `var`, or `ms_var`. |
| df | DataFrame |  | Long-format panel DataFrame (with time index, id_col, target_col) or wide-format DataFrame
(with time index and target_cols) plus optional exogenous features. |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of time steps in each forecast evaluation test set. |
| eval_metric | Callable |  | Metric function to minimize (e.g. MAE, RMSE, MAPE, MASE). |
| param_space | Dict |  | Dictionary where each value is a callable taking an Optuna `trial`. |
| target_series | Union | None | Specific series ID(s) or target column(s) to optimize the evaluation metric for.
- If a single string, optimizes accuracy ONLY for that series/target.
- If a list of strings, optimizes average accuracy across those specified series/targets.
- If None (default), optimizes average accuracy across ALL series/targets. |
| step_size | int | 1 | Step size between CV folds. |
| eval_num | int | 100 | Number of Optuna trials. |
| multivariate | bool | False | If True, uses TPE (Tree-structured Parzen Estimator) sampler for multivariate hyperparameter optimization. If False, uses default Independent sampler. |
| warm_up_steps | int | 10 | Number of initial CV folds before trial pruning begins. |
| startup_trials | int | 5 | Number of complete trials evaluated before enabling the pruner. |
| ref_series_id | Union | None | Series ID used for date-based time index splitting in panel models (defaults to series with shortest length). |
| transform_back | bool | True | If True, transforms the forecasted values back to the original scale. False is recommended when there are many series with different scales, as it may lead to large errors in the evaluation metric when target scaler is passed to model. |
| verbose | bool | False | If True, prints progress for every Optuna trial. |
| random_state | Union | 42 |  |
| **Returns** | **Tuple** |  | **(best_hparams, best_lags, other_forecaster_args)** |

## Feature selection methods for univariate time series models

In [22]:
#| echo: false
render_api(forward_feature_selection)
render_api(backward_feature_selection)

### peshbeen.model_selection.forward_feature_selection

```python
forward_feature_selection(
    model: object,
    df: pandas.core.frame.DataFrame,
    cv_split: int,
    H: int,
    metric: Callable,
    step_size: int | None = None,
    lags_to_consider: int | None = None,
    candidate_features: List[str] | None = None,
    transformations: List | None = None,
    starting_lags: List[int] | None = None,
    starting_transforms: List | None = None,
    best_start_score: float | None = None,
    improve_rate_threshold: float = 0.0,
    warm_up_steps: int = 10,
    verbose: bool = False
)
```

Forward stepwise feature selection for `ml_forecaster` models.

At each iteration every remaining candidate (lag, exogenous column, or
lag-transform) is tested individually by adding it to the current best
feature set.  The candidate that produces the largest cross-validation
improvement is permanently added.  The loop continues until no remaining
candidate improves any of the evaluation metrics.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in. |
| df | DataFrame |  | Full training DataFrame. Must contain the target column and any candidate exogenous columns. |
| cv_split | int |  | Number of time-series cross-validation folds. |
| H | int |  | Forecast horizon (test window size for each fold). |
| metric | Callable |  | One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously. |
| step_size | Union | None | Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds — consistent with the default behaviour of `ml_forecaster.cross_validate`. |
| lags_to_consider | Union | None | Consider lags `1, 2, ..., lags_to_consider` as candidates.  If `None`, lag selection is skipped. |
| candidate_features | Union | None | Column names in `df` that are exogenous feature candidates.  The function never modifies this list.  If `None`, exogenous feature selection is skipped. |
| transformations | Union | None | Lag-transform objects to test as candidates (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  The function never modifies this list.  If `None`, transform selection is skipped. |
| starting_lags | Union | None | Lags to include in the initial feature set before the search begins. These are *not* candidates — they are always included.  Must be a list (e.g. `[1]` or `[1, 2, 3]`). |
| starting_transforms | Union | None | Lag-transform objects to include in the initial feature set before the search begins.  Must be a list. |
| best_start_score | Union | None | Changed typing to float |
| improve_rate_threshold | float | 0.0 | Minimum improvement rate required for a candidate to be accepted. For example, `0.05` means the new score must be at least 5% better than the current best score to be accepted. |
| warm_up_steps | int | 10 | Number of CV folds to evaluate before pruning candidates that perform worse than the current best score. This allows the function to gather enough performance data for reliable pruning decisions. |
| verbose | bool | False | Print a message each time a candidate is accepted. |
| **Returns** |  |  | **A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the selected features.** |

### peshbeen.model_selection.backward_feature_selection

```python
backward_feature_selection(
    model: object,
    df: pandas.core.frame.DataFrame,
    cv_split: int,
    H: int,
    step_size: int | None = None,
    metrics: Callable | List[Callable] = None,
    lags_to_consider: List[int] | None = None,
    candidate_features: List[str] | None = None,
    transformations: List | None = None,
    verbose=False
)
```

Backward stepwise feature selection for `ml_forecaster` models.

Starts with the full feature set (all provided lags, exogenous columns, and
lag-transforms) and at each iteration tries removing each current feature
individually.  The feature whose removal produces the largest cross-validation
improvement is permanently dropped.  The loop continues until no remaining
feature can be removed without hurting any of the evaluation metrics.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in. |
| df | DataFrame |  | Full training DataFrame. Must contain the target column and any candidate exogenous columns. |
| cv_split | int |  | Number of time-series cross-validation folds. |
| H | int |  | Forecast horizon (test window size for each fold). |
| step_size | Union | None | Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds. |
| metrics | Union | None | One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a feature is only removed when doing so improves **all** metrics simultaneously. |
| lags_to_consider | Union | None | Lags to include in the initial feature set and test for removal (e.g. `[1, 2, 3, 4]`).  If `None`, no lag removal is attempted. |
| candidate_features | Union | None | Column names in `df` that start in the model and are tested for removal.  If `None`, exogenous feature removal is skipped. |
| transformations | Union | None | Lag-transform objects that start in the model and are tested for removal (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  If `None`, transform removal is skipped. |
| verbose | bool | False | Print a message each time a feature is removed. |
| **Returns** |  |  | **A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the surviving features after backward selection.** |

# Transformations

In [23]:
#| hide
from peshbeen.transformations import rolling_mean, rolling_std, rolling_quantile, rolling_min, rolling_max, expanding_mean, expanding_std, expanding_quantile, fourier_terms

In [24]:
#| echo: false
render_api(fourier_terms)

### peshbeen.transformations.fourier_terms

```python
fourier_terms(
    index: 'Union[pd.Index, tuple]',
    period: 'Union[int, float]',
    num_terms: 'int',
    frequency: 'Optional[str]' = None,
    t_start: 'Optional[int]' = None
)
```

Generate Fourier terms for a given index or (start, end) tuple.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| index | Union[pd.Index, tuple] |  | Either a pandas Index directly (recommended), or a (start, end) tuple of integers or datetime strings. |
| period | Union[int, float] |  | The period of the seasonality (e.g., 365.25/7 for weekly yearly seasonality). |
| num_terms | int |  | The number of Fourier term pairs (sin + cos) to generate. |
| frequency | Optional[str] | None | Frequency string (e.g., "W-SAT", "D", "M", "W"). Only relevant when index is a (start, end) tuple. |
| t_start | Optional[int] | None | Starting position of t. Only used when index is a (start, end) tuple. Use len(train_index) to ensure continuity between train and test. |
| **Returns** | **pd.DataFrame** |  | **DataFrame of Fourier terms aligned to the provided index.** |

In [25]:
#| echo: false
render_api(rolling_mean.__init__)
render_api(rolling_std.__init__)
render_api(rolling_quantile.__init__)
render_api(rolling_min.__init__)
render_api(rolling_max.__init__)
render_api(expanding_mean.__init__)
render_api(expanding_std.__init__)
render_api(expanding_quantile.__init__)

### peshbeen.transformations.rolling_mean

```python
rolling_mean(
    window_size: 'int',
    shift: 'int' = 1,
    min_samples: 'int' = 1
)
```

A class to compute the rolling mean of a time series with specified window size and shift.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| shift | int | 1 | The number of periods to shift the data before applying the rolling mean (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

### peshbeen.transformations.rolling_std

```python
rolling_std(
    window_size: 'int',
    shift: 'int' = 1,
    min_samples: 'int' = 1
)
```

A class to compute the rolling standard deviation of a time series with specified window size and shift.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| shift | int | 1 | The number of periods to shift the data before applying the rolling standard deviation (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

### peshbeen.transformations.rolling_quantile

```python
rolling_quantile(
    window_size: 'int',
    quantile: 'float',
    shift: 'int' = 1,
    min_samples: 'int' = 1
)
```

A class to compute the rolling quantile of a time series with specified window size, quantile, and shift.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| window_size | int |  | The size of the rolling window. |
| quantile | float |  | The quantile to compute (between 0 and 1). |
| shift | int | 1 | The number of periods to shift the data before applying the rolling quantile (default is 1). |
| min_samples | int | 1 | The minimum number of observations in the window required to have a value (default is 1). |
| **Returns** |  |  |  |

... (6 outputs left out)

### peshbeen.transformations.expanding_std

```python
expanding_std(
    shift: 'int' = 1
)
```

A class to compute the expanding standard deviation of a time series with specified shift.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| shift | int | 1 | The number of periods to shift the data before applying the expanding standard deviation (default is 1). |
| **Returns** |  |  |  |

### peshbeen.transformations.expanding_quantile

```python
expanding_quantile(
    shift: 'int' = 1,
    quantile: 'float' = 0.5
)
```

A class to compute the expanding quantile of a time series with specified shift and quantile.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| shift | int | 1 | The number of periods to shift the data before applying the expanding quantile (default is 1). |
| quantile | float | 0.5 | The quantile to compute (between 0 and 1) (default is 0.5). |
| **Returns** |  |  |  |